# RBS Finance Dashboard – Colab Launcher

依序執行 Cell 1 → 2 → 3 → 4，最後會印出公開網址。

| Cell | 功能 |
|------|------|
| 1 | 安裝套件 |
| 2 | 掛載 Google Drive |
| 3 | 從 GitHub 同步最新 app.py / rbs_lib.py |
| 4 | 啟動 Streamlit + pyngrok，印出公開網址 |

In [ ]:
# ── CELL 1：安裝套件 ─────────────────────────────────────────────
!pip -q install streamlit yfinance numpy pandas matplotlib seaborn \
               scikit-learn scipy adjustText feedparser newspaper3k \
               requests plotly openai

# 安裝 cloudflared（取代 pyngrok，不需要 token，無 403 問題）
import subprocess, pathlib
cf_bin = pathlib.Path('/usr/local/bin/cloudflared')
if not cf_bin.exists():
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', str(cf_bin)
    ], check=True)
    cf_bin.chmod(0o755)
    print('✅ cloudflared 安裝完成')
else:
    print('✅ cloudflared 已存在')

print('✅ 套件安裝完成')

In [ ]:
# ── CELL 2：掛載 Google Drive ────────────────────────────────────
from google.colab import drive
from pathlib import Path
import sys

# 先 mount，再建目錄
drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = Path('/content/drive/MyDrive/RBS_app')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

if str(DRIVE_BASE) not in sys.path:
    sys.path.insert(0, str(DRIVE_BASE))

print(f'✅ Google Drive 掛載完成')
print(f'   工作目錄：{DRIVE_BASE}')

In [ ]:
# ── CELL 3：從 GitHub 同步最新程式碼 ─────────────────────────────
import urllib.request

BRANCH = 'claude/optimize-analysis-dashboard-NZUKB'
REPO   = 'RoyARVSA/RBS_claude_finance'
FILES  = ['app.py', 'rbs_lib.py']

for fname in FILES:
    url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{fname}'
    dst = DRIVE_BASE / fname
    try:
        urllib.request.urlretrieve(url, dst)
        size_kb = dst.stat().st_size / 1024
        print(f'  ✅  {fname:<12} ({size_kb:.1f} KB)')
    except Exception as e:
        if dst.exists():
            print(f'  ⚠️  {fname}: 下載失敗，使用本地快取 ({e})')
        else:
            raise RuntimeError(f'{fname} 下載失敗且無快取：{e}')

print('✅ 同步完成')

In [ ]:
# ── CELL 4：啟動 Streamlit + cloudflared ────────────────────────
import subprocess, socket, time, sys, re
from pathlib import Path

# ── 設定 ──────────────────────────────────────────────────────
PORT     = 8501
APP_PATH = DRIVE_BASE / 'app.py'
LOG_PATH = DRIVE_BASE / 'streamlit.log'

# ── 清除舊進程 ────────────────────────────────────────────────
subprocess.run(['fuser', '-k', f'{PORT}/tcp'], capture_output=True)
time.sleep(0.5)

# ── 啟動 Streamlit（log 寫入檔案，方便 debug）────────────────
print(f'🚀 Starting Streamlit on port {PORT}…')
_log = open(LOG_PATH, 'w')
st_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', str(APP_PATH),
        '--server.port',                str(PORT),
        '--server.address',             '0.0.0.0',
        '--server.headless',            'true',
        '--server.enableCORS',          'false',
        '--server.enableXsrfProtection','false',
        '--browser.gatherUsageStats',   'false',
    ],
    stdout=_log, stderr=_log,
)

# ── 等待就緒（最多 60 秒）────────────────────────────────────
t0 = time.time()
while time.time() - t0 < 60:
    try:
        socket.create_connection(('127.0.0.1', PORT), 1).close()
        print(f'✅ Streamlit ready ({time.time()-t0:.1f}s)')
        break
    except OSError:
        time.sleep(0.3)
else:
    print('❌ Streamlit 60s 內未啟動，查看 log：')
    print(Path(LOG_PATH).read_text()[-800:])
    raise RuntimeError('Streamlit failed to start')

# ── cloudflared tunnel ────────────────────────────────────────
print('🔗 Creating cloudflared tunnel…')
CF_LOG = '/tmp/cloudflared.log'
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=open(CF_LOG, 'w'),
    stderr=subprocess.STDOUT,
)

# 等待 trycloudflare.com 網址出現（最多 30 秒）
pub = None
for _ in range(60):
    time.sleep(0.5)
    log_text = open(CF_LOG).read()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', log_text)
    if match:
        pub = match.group(0)
        break

if not pub:
    print('❌ 無法取得 cloudflared 網址，查看 log：')
    print(open(CF_LOG).read()[-800:])
    raise RuntimeError('cloudflared tunnel failed')

# ── 顯示結果 ──────────────────────────────────────────────────
from IPython.display import display, HTML
print()
print('╔' + '═'*54 + '╗')
print(f'║  🌐  Public URL : {pub}')
print(f'║  📂  App        : {APP_PATH}')
print(f'║  📋  Log        : {LOG_PATH}')
print('╠' + '═'*54 + '╣')
print('║  To stop  →  st_proc.terminate(); cf_proc.terminate()')
print('╚' + '═'*54 + '╝')
display(HTML(f'<a href="{pub}" target="_blank" style="font-size:16px">👉 點此開啟 Dashboard</a>'))